# Seeing Why: exact evidence maps, interactive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stein-research/seeing-why/blob/main/demo/demo.ipynb)

Companion to the paper *Seeing Why: Exact Evidence Maps for Quantum and Optical Image Classifiers*. The classifiers
here have no trained classical network: a picture is encoded as amplitudes, passes a Fourier-plane mask and either
four phase plates (light) or a brickwall circuit (qubits), and the measured probabilities are read with the Born
headroom readout. Because the model is linear in the input amplitudes up to the measurement, the evidence of every
pixel has a closed form and adds up to the class's probability above its share.

Runs on a CPU (no GPU needed), with the trained models of the paper (Table 1, seed 0). Run the cells in order.

In [ ]:
# Get the code (in Colab: a fresh copy of the repository; when opened from a clone: that clone)
import os, sys
if os.path.exists("ui.py"):
    ROOT = ".."
elif os.path.exists("demo/ui.py"):
    ROOT = "."
else:
    !git clone -q --depth 1 https://github.com/stein-research/seeing-why
    ROOT = "seeing-why"
sys.path.insert(0, os.path.join(ROOT, "demo"))
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
%matplotlib inline
import ui, seeing_why as sw

## 1. Pick a picture, see why

Choose a **model**, a **class** and one of its 30 test pictures (or **upload your own photo**; it is cut to the
centre square and resized to 32 x 32 like the test set). **explain** chooses the class whose evidence is drawn:
the predicted one, or any other ("why not leopard?"). Red pixels push that class above its share, blue ones pull it
below; near-zero evidence is left transparent.

**delete top** runs the deletion test of the paper: the pixels with the highest evidence lose their contrast (they
take the picture's own mean colour) and the model is run again. If the map is faithful, a few percent of pixels are
enough to change the answer.

In [ ]:
ui.demo()

## 2. Plain amplitudes against the mean plane

The same picture through the four models of the paper. With plain amplitudes the light, and with it the evidence,
follows the bright parts of the picture. The mean plane removes each picture's own mean from each colour plane and
carries the means in a fourth plane, so dark details weigh as much as bright ones and the evidence moves onto the
object (the ketch's dark hull, for example).

In [ ]:
ui.demo_compare()

## 3. Measured on a quantum computer

Ten test pictures, one per class, were run on IBM's ibm_kingston. Each 2 x 2 evidence map was measured with the
polarization identity (nine circuits per picture, all in one job). Left: the exact map of the simulation; right: the
chip's. Every job is kept in `ibm_jobs/` (job id, calibration, qubits used, circuits as sent, raw counts).

In [ ]:
ui.demo_chip()

## 4. Without widgets

The same functions work in plain Python:

In [ ]:
x, y = sw.test_pictures()
run = sw.load("phase plates (row D)")
p = sw.classify(run, x)                         # BHR class probabilities of all 300 test pictures
print(f"test accuracy {100 * (p.argmax(1) == y).mean():.1f} %")
e = sw.evidence(run, x[:1], [int(y[0])])[0]      # 32 x 32 evidence for the true class of the first picture
print(e.shape, e.sum())